Some routine setup. *Don't forget to enable your secret in the Colab UI!*

In [ ]:
!pip install -q -U langchain langchain-google-genai

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Tools & Memory: Agent Backbones

## The Problem

ACME Clothing finally made their first chatbot and tested it with their customers. It knows all sorts of things about the company, but it wasn't ready for Shelley.

Your customer, Shelley, messages your retail chatbot: *"Do you have the blue T-Shirt in stock?"* The agent doesn't yet have access to real-time inventory. Try it and see what happens:

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """
ACME Clothing is literally the best retail clothing supplier.
Our motto: Buy now, never regret.

We supply all your clothing needs:
- Hoodies
- T-Shirts

They vary on whether they are in stock or not, but we have them eventually.

All you could ever hope for.
"""

agent = create_agent(
    model="google_genai:gemini-flash-lite-latest",
    system_prompt=SYSTEM_PROMPT
)

In [ ]:
# Critical Insights:
#   1. Without use of the 'check_inventory' tool below the response will be ignorant to inventory stock data.

from langchain.messages import HumanMessage

prompt = HumanMessage("Do you have the blue T-Shirt in stock?")

response = agent.invoke({"messages": [prompt]})
print(response["messages"][-1].text)

AI will probably guess, say it doesn't know, or make something up. Either way, the agent has no way to actually know. That's a problem!

## Adding Tools

Fortunately, you already have a function that accesses inventory, but how do you give it to the agent?

Check the [documentation](https://docs.langchain.com/oss/python/langchain/tools) or follow along with the instructor for the answer.

In [ ]:
# Critical changes:
#   1. Added the import statement for 'tool'
#   2. Added "@tool" decorator
#   3. Added tool description inside the method

from langchain.tools import tool

@tool
def check_inventory(item: str, color: str) -> str:
    """Get stuff."""
    fake_stock = {
        ("hoodie", "blue"): 4,
        ("hoodie", "black"): 12,
        ("t-shirt", "blue"): 30,
    }

    if (item.lower(), color.lower()) in fake_stock:
        return "We're in stock!"
    else:
        return "We're out of stock..."

    if (color.lower()) in fake_stock[color]:
      return "We have items of that color!"
    else:
      return "We don't have any items of that color..."

In [ ]:
# Notice of Failure:
#   1. Regular functions/methods are DIFFERENT from tools.
#   2. These print statments...
#      - WILL work with regular function calls of the 'check_inventory' method...
#      - BEFORE the method is provided a decorator ('@tool') one line above itself.
#      - This change is meant to convert the method into a tool for use by AI Agents.


print(check_inventory("hoodie", "blue"))  # Failure: 1
print(check_inventory("t-shirt", "gray")) # Failure: 2

In [ ]:
# Observation:
#   1. Because we changed the 'check_inventory' method into a tool...
#     - The AI Agent can make full use of the tool.
#     - The answer is now CORRECT!

agent = create_agent(
    model="google_genai:gemini-flash-lite-latest",
    tools=[check_inventory],
    system_prompt=SYSTEM_PROMPT
)

response = agent.invoke({"messages": [prompt]})
print(response["messages"][-1].text)

### Check Your Understanding

- How does the function name and docstring change how your function is called?
- How does a `@tool` function differ from a standard Python function?
- What is required to make a tool? What is optional, but best practice? (You might try removing components to see)

## But Wait There's More: Memory

You Did It! But Shelley has one more trick up her sleeve. She asks a follow-up question: "How about gray?" Is your agent ready for such a complicated question? Run the code and see what happens.

In [ ]:
# Unexpected Question Angles:
#   1. If the user prompts the Agent with a semi-complete question, the Agent needs tools that guide it through how to begin answering that question. Then, the Agent's predictive capacity allows for the smoothing out of these types of challenges.

prompt = HumanMessage("How about black?")

response = agent.invoke({"messages": [prompt]})
print(response["messages"][-1].text)

Whoa, what happened? Take a look at it's entire context history.

In [ ]:
# How did the Agent get the answer?
#   - It looks like the Agent...
#     1. ingested "black" and "color",
#     2. used the 'get-it' tool and found the color first,
#     3. then it pulled "hoodies" by association,
#     4. finally, it asked if the user wanted another item in black? This would be usefull if the stock had more than hoodies in black, or had many items in black which is more likely across store's actual inventory and offerings as black is a very common color.

import pprint
pprint.pprint(response)

Aside from maybe needing a better tool definition, the agent forgot its conversation history. See if you can fill in the blanks.

See the [documentation](https://docs.langchain.com/oss/python/langchain/short-term-memory) if you need help.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-flash-lite-latest",
    tools=[check_inventory],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver()
)

In [ ]:
thread_config = {"configureable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage("How about black?")]},
    thread_config,
)

print(response["messages"][-1].text)

In [ ]:
pprint.pprint(response)

## Choose Your Own Adventure

Now that you have the basics, explore! Here are a few things to try:

- What happens if you run the prompt (`agent.invoke()`) in the above code chunk again? And again? Check with `pprint()` to see the conversation history
- What happens if you change the `thread_id`?
- What else can you do with this agent as-is? What would be useful for an online retailer? In-store retailer?
- Can you give the agent a more useful function? Maybe add number of items in stock?
- How can you constrain tool ouput to be a little more consistent? (hint: you can check the documentation)
- What else is in the documentation that might be useful?
- Can you add multiple messages in the brackets inside `.invoke()` (hint: check the documentation for what goes in [messages](https://docs.langchain.com/oss/python/langchain/messages))


## Apply It: a fresh problem, on your own

A customer messages your shipping company: *"Where is shipment TRK-4821?"* Your agent doesn't have shipment data yet, and after it answers, the customer will ask a follow-up ("Has it left the warehouse yet?") that only makes sense if the agent remembers the tracking ID.

Build this yourself, start to finish:

1. Turn `get_shipment_status(tracking_id: str) -> str` into a tool tool with a clear docstring, using fake data of your own
2. Wire it into an agent and confirm the first question gets answered correctly
3. Ask the follow-up question, carrying the conversation forward the same way you just did for Shelley
4. Confirm the agent correctly resolves what "it" refers to *and* calls the tool again to answer

In [ ]:
def get_shipment_status(tracking_id: str) -> str:
    """Get the current shipping status for a given tracking ID."""
    fake_shipment_data = {
        "TRK-4821": "Your shipment TRK-4821 is currently in transit and is expected to arrive within 2-3 business days.",
        "TRK-5555": "Your shipment TRK-5555 has left the warehouse and is on its way.",
        "TRK-9000": "Your shipment TRK-9000 is awaiting pickup at the warehouse."
    }
    return fake_shipment_data.get(tracking_id, "Tracking ID not found. Please check your tracking ID and try again.")